# Random Coarsen Uniform
## Introduction

### Description
 
Test of multigrid coarsen up to 4. An arbitrary number of coarsen operator is possible, as we lifted the restriction hard-coded in TRUST multigrid. A comparison on the pressure gradient field is performed using `compare_lata`.

The validation test case is a rectangular grid of 48x48x24 elements. The divisors are [1, 2, 3, 4, 6, 8, 12, 24] and 48. We compare 1, 2 and 3 levels of coarsen operators. The validation is performed on the pressure gradient. The solver must give the same result up to the threshold defined in the dataset.

In [ ]:
from trustutils import run 
from pathlib import Path
from itertools import product, combinations, permutations
from sympy import divisors
import os
from IPython.display import Markdown as md

run.TRUST_parameters("1.9.4_beta")

## Computation preparation

In [ ]:
run.reset()
run.initBuildDirectory() 
jddrootname = "rcu"

def copy_init_lata(casename):
    run.executeCommand(f"mkdir -p {run.BUILD_DIRECTORY}/{casename}")
    run.executeCommand(f"cp init.tar.gz {run.BUILD_DIRECTORY}/{casename}/init.tar.gz")
    run.executeCommand(f"cd {run.BUILD_DIRECTORY}/{casename} && tar -xf init.tar.gz")
    

#### Grid parameters

We use $ni = 48$ and $nk = 24$ to have lots of divisors.

In [ ]:
ni = 48
nk = 24

## Test of different levels of multigrid

### One level of Coarsen Operator

Test of all permutations between 1 and 4.

In [ ]:
print(f" It leads to {len(list(product(range(1, 5), repeat=3)))} calculations")

Computations are stored in `build/1-LEVEL/COARSEN-XYZ` with `X`, `Y` and `Z` the coarsen divisor.

In [ ]:

exclu=False
for arg in product(range(1, 5), repeat=3):
    casename = f"1-LEVEL/COARSEN-{arg[0]}{arg[1]}{arg[2]}"
    mydict = {
        "coarsen_operators": "coarsen_operators 1",
        "mycoarsen": f"Coarsen_Operator_Uniform {{ coarsen_i {arg[0]} coarsen_j {arg[1]} coarsen_k {arg[2]} }}"
    }
    run.addCaseFromTemplate(f"{jddrootname}.data",
                            dic=mydict, 
                            nbProcs=1,
                            targetDirectory=casename,
                            excluNR=exclu,
                            targetData=f"{jddrootname}-{arg[0]}{arg[1]}{arg[2]}.data")
    copy_init_lata(casename)
    # so that we only add 1 test per level
    exclu=True

### Two levels of Coarsen Operators

We create combinations of divisors for the 2-LEVEL test. All test are stored in `build/2-LEVEL/COARSEN-NiXY-NkXY` as we use the same divisors for `I` and `J`.

In [ ]:
def compute_combinations_2(nval):
    return [(a, b) for (a, b) in combinations(divisors(nval), 2) if a*b == nval]

In [ ]:
combi_ni = compute_combinations_2(ni)
combi_nk = compute_combinations_2(nk)

In [ ]:
md(f"The possible combinations of 2-LEVEL coarsen operators for ni are {combi_ni}, and {combi_nk} for nk. It leads to {len(combi_ni)*len(combi_nk)} possibilities.")

We choose `(3, 16)` and `(6, 8)` for `ni` and `(2, 12)` and `(4, 6)` for `nk`.

In [ ]:
listNi = [(3, 16), (6, 8)]
listNk = [(2, 12), (4, 6)]
exclu=False
for combiNi in listNi:
    for combiNk in listNk:
        for argNi in permutations(combiNi):
            for argNk in permutations(combiNk):
                casename = f"2-LEVEL/COARSEN-Ni{argNi[0]}{argNi[1]}-Nk{argNk[0]}{argNk[1]}"
                mydict = {
                    "coarsen_operators": "coarsen_operators 2",
                    "mycoarsen": f"Coarsen_Operator_Uniform {{ coarsen_i {argNi[0]} coarsen_j {argNi[0]} coarsen_k {argNk[0]} }}\nCoarsen_Operator_Uniform {{ coarsen_i {argNi[1]} coarsen_j {argNi[1]} coarsen_k {argNk[1]} }}"
                }
                run.addCaseFromTemplate(f"{jddrootname}.data",
                                        dic=mydict, 
                                        nbProcs=1,
                                        targetDirectory=casename,
                                        excluNR=exclu,
                                        targetData=f"{jddrootname}-Ni{argNi[0]}{argNi[1]}-Nk{argNk[0]}{argNk[1]}.data")
                copy_init_lata(casename)
                # so that we only add 1 test per level
                exclu=True

### Three levels of Coarsen Operators

In [ ]:
def compute_combinations_3(nval):
    return [(a, b, c) for (a, b, c) in combinations(divisors(nval), 3) if a*b*c == nval]

In [ ]:
combi_ni = compute_combinations_3(ni)
combi_nk = compute_combinations_3(nk)

In [ ]:
md(f"The possible combinations of 3-LEVEL coarsen operators for ni are {combi_ni}, and {combi_nk} for nk. It leads to {len(combi_ni)*len(combi_nk)} possibilities.")

We choose `(2, 3, 8)` and `(2, 3, 4)`

In [ ]:
listNi = [(2, 3, 8)]
listNk = [(2, 3, 4)]
exclu=False
for combiNi in listNi:
    for combiNk in listNk:
        for argNi in permutations(combiNi):
            for argNk in permutations(combiNk):
                casename = f"3-LEVEL/COARSEN-Ni{argNi[0]}{argNi[1]}{argNi[2]}-Nk{argNk[0]}{argNk[1]}{argNk[2]}"
                mydict = {
                    "coarsen_operators": "coarsen_operators 3",
                    "mycoarsen": f"Coarsen_Operator_Uniform {{ coarsen_i {argNi[0]} coarsen_j {argNi[0]} coarsen_k {argNk[0]} }}\nCoarsen_Operator_Uniform {{ coarsen_i {argNi[1]} coarsen_j {argNi[1]} coarsen_k {argNk[1]} }}\nCoarsen_Operator_Uniform {{ coarsen_i {argNi[2]} coarsen_j {argNi[2]} coarsen_k {argNk[2]} }}"
                }
                run.addCaseFromTemplate(f"{jddrootname}.data",
                                        dic=mydict, 
                                        nbProcs=1,
                                        targetDirectory=casename,
                                        excluNR=exclu,
                                        targetData=f"{jddrootname}-Ni{argNi[0]}{argNi[1]}{argNi[2]}-Nk{argNk[0]}{argNk[1]}{argNk[2]}.data") 
                copy_init_lata(casename)
                # so that we only add 1 test per level
                exclu=True

## Run cases

In [ ]:
run.runCases()

## Computer Performance

In [ ]:
table = run.tablePerf()
table.index = [zzz.split("/")[-1] for zzz in table.index]
table = table.drop("system", axis=1).drop("host", axis=1)
table

## Compare Lata 

### Choose reference case

Table legend:
- Blue: Reference case
- Black: Same result
- Red: Different result

In [ ]:

import re
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = [10, 4]
import numpy as np

def compare_lata():
    os.system(f'cd {run.BUILD_DIRECTORY} && bash compare-lata.bash')

run.saveFileAccumulator("compare_lata_done")
compare_lata()

In [ ]:
def color_negative_red(val):
    color = 'red' if str(val) in fail else 'blue' if str(val) == ref else 'black'
    return 'color: %s' % color

def find_fail():
    file_path = f'1-LEVEL/COMPARELATA/non-zero-diff.log'
    run.saveFileAccumulator(file_path)
    pattern = re.compile(r'(\d{3})')
    fail = []
    with open(f"{run.BUILD_DIRECTORY}/{file_path}", 'r') as file:
        for line in file:
            match = pattern.search(line)
            if match:
                fail.append(match.group(1))
                # print(f'Case {match.group(1)} error exceeds threshold 1e-5 (expected to happen)')
    return fail 

def table_comp():
    coarsi1 = [] 
    coarsi2 = []
    coarsi3 = []
    coarsi4 = []

    bool1 = []
    bool2 = []
    bool3 = []
    bool4 = []
    
    fail = find_fail()
    
    for arg in product(range(1, 5), repeat=3):
        case = f"{arg[0]}{arg[1]}{arg[2]}"
        if arg[0] == 1:
            coarsi1.append(int(case))
            if str(case) in fail:
                bool1.append(True)
            else:
                bool1.append(False)
        if arg[0] == 2:
            coarsi2.append(int(case))
            if str(case) in fail:
                bool2.append(True)
            else:
                bool2.append(False)
        if arg[0] == 3:
            coarsi3.append(int(case))
            if str(case) in fail:
                bool3.append(True)
            else:
                bool3.append(False)
        if arg[0] == 4:
            coarsi4.append(int(case))
            if str(case) in fail:
                bool4.append(True)
            else:
                bool4.append(False)
        
    df = pd.DataFrame({'coarsi1': coarsi1,
                       'coarsi2': coarsi2,
                       'coarsi3': coarsi3,
                       'coarsi4': coarsi4,
                      })
    
    return df

#### Reference case: 111

In [ ]:
ref = '111'
fail = find_fail()
df = table_comp()
m = df.style.map(color_negative_red)
print("In blue, reference case (111)")
print("In red, cases where error exceeds 1e-5")
display(m)

#### Plot the error

In [ ]:
def find_maximal_error(file_path):
    p=r'Maximal relative error encountered : ([\d.e+-]+)'
    error_pattern = re.compile(p)
    run.saveFileAccumulator(file_path)
    with open(f"{run.BUILD_DIRECTORY}/{file_path}", 'r') as file:
        for line in file:
            match = error_pattern.search(line)
            if match:
                error_value = match.group(1)
                error= float(error_value)

                return error
    raise RuntimeError("pattern not found: ", p)

def list_error():
    L_case = []
    L_error = []
    for arg in product(range(1, 5), repeat=3):
        case = f"{arg[0]}{arg[1]}{arg[2]}"
        L_case.append(case)
        error = find_maximal_error(f'1-LEVEL/COMPARELATA/compare-lata-{case}.log')
        L_error.append(error)
    sorted_L_error, sorted_L_case = zip(*sorted(zip(L_error, L_case)))
    plt.plot(sorted_L_case,sorted_L_error,'--x', label="maximal relative error to ref (111)")
    plt.plot(sorted_L_case,np.full(len(L_case), 1e-5), label="compare_lata threshold")
    # plt.hlines(10**-5,L_case[0],L_case[-1],color='r')
    plt.xticks(rotation=90)
    plt.xlabel('Coarsen Operator')
    plt.ylabel('Maximal relative error')
    plt.legend(loc="upper right")
    plt.grid()

list_error()

### 3-Level 

In [ ]:
def color_negative_red_l3(val):
    color = 'red' if str(val) in fail else 'blue' if str(val)==ref else 'black'
    return 'color: %s' % color

def find_fail_l3():
    file_path = f'3-LEVEL/COMPARELATA/non-zero-diff.log'
    run.saveFileAccumulator(file_path)
    pattern = re.compile(r'Ni(\d+)-Nk(\d+)')
    fail = []
    with open(f"{run.BUILD_DIRECTORY}/{file_path}", 'r') as file:
        for line in file:
            match = pattern.search(line)
            ni_number = int(match.group(1))
            nk_number = int(match.group(2))
            fail.append(f"{ni_number}-{nk_number}")
            # print(f'Case {ni_number}-{nk_number} error exceeds threshold 1e-5 (expected to happen)')
    return fail 

def table_comp_l3():
    coarsi2 = [] 
    coarsi3 = []
    coarsi8 = []

    bool2 = []
    bool3 = []
    bool8 = []
    
    fail = find_fail_l3()
    listNi = [(2, 3, 8)]
    listNk = [(2, 3, 4)]
    for combiNi in listNi:
        for combiNk in listNk:
            for argNi in permutations(combiNi):
                for argNk in permutations(combiNk):
                    case = f"{argNi[0]}{argNi[1]}{argNi[2]}-{argNk[0]}{argNk[1]}{argNk[2]}"
                    if argNi[0] == 2:
                        coarsi2.append(case)
                        bool2.append(case in fail)
                    if argNi[0] == 3:
                        coarsi3.append(case)
                        bool3.append(case in fail)
                    if argNi[0] == 8:
                        coarsi8.append(case)
                        bool8.append(case in fail)

        
    df = pd.DataFrame({'coarsi2': coarsi2,
                       'coarsi3': coarsi3,
                       'coarsi8': coarsi8
                      })
    
    return df

In [ ]:
fail = find_fail_l3()
df = table_comp_l3()
m = df.style.map(color_negative_red_l3)
print("In red, cases where error exceeds 1e-5 (relative to ref 1-level coarsen 111)")
display(m)

## Time per timestep

From a few tests, these graphs do not seem very reliable to rank the performance of the different coarsen operators. The results seem to vary greatly between different runs, even in sequential mode.

In [ ]:
def find_timeperstep(file_path):
    pattern = re.compile(r'Average time per time step:     .*     ([\d.e+-]+)')
    run.saveFileAccumulator(file_path)
    with open(f"{run.BUILD_DIRECTORY}/{file_path}", 'r') as file:
        for line in file:
            match = pattern.search(line)
            if match:
                value = float(match.group(1))
                return value
    raise RuntimeError("Average time per step not found")

ref_time=None
def list_timeperstep_L1():
    global ref_time
    L_case_1 = []
    L_timeperstep_1 = []
    L_error_1 = []
    for arg in product(range(1, 5), repeat=3):
        case = f"{arg[0]}{arg[1]}{arg[2]}"
        L_case_1.append(case)
        timeperstep = find_timeperstep(f'1-LEVEL/COARSEN-{case}/rcu-{case}.TU')
        L_timeperstep_1.append(timeperstep)
        error = find_maximal_error(f'1-LEVEL/COMPARELATA/compare-lata-{case}.log')
        L_error_1.append((error)*timeperstep)
        if case=="111":
            ref_time=timeperstep
    plt.figure()
    sorted_L_timeperstep_1, sorted_L_case_1, sorted_error_1 = zip(*sorted(zip(L_timeperstep_1, L_case_1, L_error_1)))
    plt.plot(sorted_L_case_1,sorted_L_timeperstep_1,'--x',label="time per step")
    plt.plot(sorted_L_case_1, np.full(len(sorted_L_case_1), ref_time), label="ref  (111)")
    plt.xticks(rotation=90)
    plt.xlabel('Coarsen Operator')
    plt.ylabel('Time per step [s]')
    plt.legend()
    plt.grid()
    plt.title("1-level: time per step")
    
    plt.figure()
    sorted_error_1, sorted_L_timeperstep_1, sorted_L_case_1 = zip(*sorted(zip(L_error_1, L_timeperstep_1, L_case_1)))
    plt.plot(sorted_L_case_1, sorted_error_1, "--o", label="error * time (lower=better)")
    plt.xticks(rotation=90)
    plt.xlabel('Coarsen Operator')
    plt.legend()
    plt.title("1-level: time per step * error (metric is biased toward 111)")
    plt.grid()
    
def list_timeperstep_L2(): 
    L_case_2 = []
    L_timeperstep_2 = []
    L_error_2 = []
    listNi = [(3, 16), (6, 8)]
    listNk = [(2, 12), (4, 6)]
    for combiNi in listNi:
        for combiNk in listNk:
            for argNi in permutations(combiNi):
                for argNk in permutations(combiNk):
                    case = f"{argNi[0]}{argNi[1]}-{argNk[0]}{argNk[1]}"
                    L_case_2.append(case)
                    timeperstep = find_timeperstep(f'2-LEVEL/COARSEN-Ni{argNi[0]}{argNi[1]}-Nk{argNk[0]}{argNk[1]}/rcu-Ni{argNi[0]}{argNi[1]}-Nk{argNk[0]}{argNk[1]}.TU')
                    L_timeperstep_2.append(timeperstep)
                    error = find_maximal_error(f'2-LEVEL/COMPARELATA/compare-lata-Ni{argNi[0]}{argNi[1]}-Nk{argNk[0]}{argNk[1]}.log')
                    L_error_2.append(error*timeperstep)
    plt.figure()    
    sorted_L_timeperstep_2, sorted_L_case_2 = zip(*sorted(zip(L_timeperstep_2, L_case_2)))
    plt.plot(sorted_L_case_2,sorted_L_timeperstep_2,'--x',label="2-LEVEL")
    plt.plot(sorted_L_case_2, np.full(len(sorted_L_case_2), ref_time), label="ref (111)")
    plt.gcf().autofmt_xdate(rotation=55)
    plt.xlabel('Coarsen Operator')
    plt.ylabel('Time per step [s]')
    plt.legend()
    plt.grid()
    plt.figure()
    

def list_timeperstep_L3(): 
    listNi = [(2, 3, 8)]
    listNk = [(2, 3, 4)]
    L_case_3 = []
    L_timeperstep_3 = []
    for combiNi in listNi:
        for combiNk in listNk:
            for argNi in permutations(combiNi):
                for argNk in permutations(combiNk):
                    case = f"{argNi[0]}{argNi[1]}{argNi[2]}-{argNk[0]}{argNk[1]}{argNk[2]}"
                    L_case_3.append(case)
                    timeperstep = find_timeperstep(f'3-LEVEL/COARSEN-Ni{argNi[0]}{argNi[1]}{argNi[2]}-Nk{argNk[0]}{argNk[1]}{argNk[2]}/rcu-Ni{argNi[0]}{argNi[1]}{argNi[2]}-Nk{argNk[0]}{argNk[1]}{argNk[2]}.TU')
                    L_timeperstep_3.append(timeperstep)
    plt.figure()
    sorted_L_timeperstep_3, sorted_L_case_3 = zip(*sorted(zip(L_timeperstep_3, L_case_3)))
    plt.plot(sorted_L_case_3,sorted_L_timeperstep_3,'--x',label="3-LEVEL")
    plt.plot(sorted_L_case_3, np.full(len(sorted_L_case_3), ref_time), label="ref (111)")
    plt.gcf().autofmt_xdate(rotation=55)
    plt.xlabel('Coarsen Operator')
    plt.ylabel('Time per step [s]')
    plt.legend()
    plt.grid()

list_timeperstep_L1()
list_timeperstep_L2()
list_timeperstep_L3()